# Fine-Tuning Qwen2.5-0.5B-Instruct for Dashboard Design Recommendations
**Master Thesis | PEFT/QLoRA | Google Colab**

## Instructions
1. Set runtime to GPU: **Runtime > Change runtime type > T4 GPU**
2. Run all cells top to bottom (Ctrl+F9 or Runtime > Run all)
3. Training takes ~15-30 minutes on T4

## What this notebook does:
- Installs all required libraries
- Writes project files directly into Colab
- Generates a synthetic training dataset (80 examples)
- Fine-tunes Qwen2.5-0.5B-Instruct with QLoRA
- Evaluates the fine-tuned model vs. base model
- Optionally saves the adapter to Google Drive

## Step 1: Check GPU & Install Libraries

In [ ]:
# Check GPU
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode == 0:
    print(r.stdout)
else:
    print('No GPU! Go to Runtime > Change runtime type > T4 GPU')


In [ ]:
# Install libraries (~3-5 minutes)
!pip install -q transformers==4.40.0 datasets peft trl bitsandbytes accelerate
!pip install -q pyyaml jsonlines evaluate
print('Installation complete!')


## Step 2: Write Project Files

In [ ]:
import os
os.makedirs('config', exist_ok=True)
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('outputs/models', exist_ok=True)
os.makedirs('outputs/predictions', exist_ok=True)
print('Directories created.')


In [ ]:
# Write config/train_config.yaml
cfg = '''
model:
  name: "Qwen/Qwen2.5-0.5B-Instruct"
  max_seq_length: 2048
  load_in_4bit: true
  load_in_8bit: false
lora:
  r: 16
  lora_alpha: 32
  lora_dropout: 0.05
  bias: "none"
  task_type: "CAUSAL_LM"
  target_modules: ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
training:
  output_dir: "./outputs/models/checkpoints"
  num_epochs: 3
  batch_size: 2
  gradient_accumulation_steps: 4
  learning_rate: 2.0e-4
  lr_scheduler: "cosine"
  warmup_ratio: 0.1
  weight_decay: 0.01
  fp16: true
  bf16: false
  logging_steps: 10
  save_steps: 50
  save_total_limit: 2
  seed: 42
  gradient_checkpointing: true
data:
  raw_dir: "./data/raw"
  train_file: "./data/processed/train.jsonl"
  val_file: "./data/processed/val.jsonl"
  test_file: "./data/processed/test.jsonl"
  num_train: 80
  num_val: 10
  num_test: 10
  seed: 42
inference:
  adapter_path: "./outputs/models/final"
  max_new_tokens: 1024
  temperature: 0.1
  top_p: 0.9
  do_sample: true
paths:
  models_dir: "./outputs/models"
  final_model_dir: "./outputs/models/final"
  logs_dir: "./outputs/logs"
  predictions_dir: "./outputs/predictions"
'''
with open('config/train_config.yaml', 'w') as f:
    f.write(cfg)
print('config/train_config.yaml written.')


In [ ]:
# Upload scripts from your local machine
# Select: 02_generate_synthetic_dataset.py, 03_prepare_dataset.py,
#         04_train_lora.py, 06_inference_finetuned_model.py,
#         07_evaluate_schema_compliance.py
from google.colab import files
print('Upload your script files now:')
uploaded = files.upload()
os.makedirs('scripts', exist_ok=True)
for fname in uploaded:
    os.rename(fname, f'scripts/{fname}')
    print(f'  Moved -> scripts/{fname}')


## Step 3: Generate Synthetic Dataset

In [ ]:
!python scripts/02_generate_synthetic_dataset.py

# Verify
for f in ['data/raw/train.jsonl','data/raw/val.jsonl','data/raw/test.jsonl']:
    if os.path.exists(f):
        with open(f) as fp: n = sum(1 for _ in fp)
        print(f'  {f}: {n} examples')
    else:
        print(f'  MISSING: {f}')


## Step 4: Prepare Dataset (Format for Training)

In [ ]:
!python scripts/03_prepare_dataset.py

# Show a sample
import json
with open('data/processed/train.jsonl') as f:
    sample = json.loads(f.readline())
print('Sample formatted text (first 500 chars):')
print(sample['text'][:500])


## Step 5: Fine-Tune with QLoRA

**Expected time**: ~15-25 minutes on T4 GPU

**Watch the loss**: Should decrease from ~2.0-3.0 to ~0.3-0.8 after 3 epochs.

If you get `CUDA out of memory`: change `batch_size: 1` in the config cell above and re-run.

In [ ]:
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {name}  (VRAM: {vram:.1f} GB)')
    if vram < 8:
        print('Low VRAM! Set batch_size: 1 in config and re-run Step 2.')
    else:
        print('VRAM sufficient for training.')
else:
    print('No GPU! Go to Runtime > Change runtime type > T4 GPU')


In [ ]:
!python scripts/04_train_lora.py


In [ ]:
# Verify model was saved
model_dir = 'outputs/models/final'
if os.path.exists(model_dir):
    print('Model saved! Files:')
    for fname in os.listdir(model_dir):
        size = os.path.getsize(os.path.join(model_dir, fname)) / 1024 / 1024
        print(f'  {fname} ({size:.1f} MB)')
else:
    print('Model directory not found. Check training output above.')


## Step 6: Inference with Fine-Tuned Model

In [ ]:
!python scripts/06_inference_finetuned_model.py


## Step 7: Evaluate Schema Compliance

In [ ]:
!python scripts/07_evaluate_schema_compliance.py


In [ ]:
# Show evaluation report
report_path = 'outputs/predictions/evaluation_report.json'
if os.path.exists(report_path):
    with open(report_path) as f:
        report = json.load(f)
    for r in report:
        m = r['metrics']
        print(f"Model: {m['model_label']}")
        print(f"  JSON parse rate:      {m['json_parse_rate']}%")
        print(f"  Schema validity rate: {m['schema_validity_rate']}%")
        print(f"  Avg latency:          {m['avg_latency_s']}s")
        print()


## Step 8: Save Model to Google Drive (Optional)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
save_path = '/content/drive/MyDrive/master-thesis/fine-tuned-model'
shutil.copytree('outputs/models/final', save_path, dirs_exist_ok=True)
print(f'Model saved to Google Drive: {save_path}')


## Troubleshooting

| Error | Fix |
|-------|-----|
| `CUDA out of memory` | Set `batch_size: 1` in config, re-run Step 2 |
| `bitsandbytes` error | Re-run the install cell |
| `FileNotFoundError: train.jsonl` | Run Steps 3 and 4 first |
| `Adapter not found` | Run Step 5 first |
| Loss not decreasing | Try `learning_rate: 5.0e-4` |
| Model outputs invalid JSON | Increase `num_epochs: 5` |

### Understanding Loss:
- Start: ~2.0-3.0 (normal)
- After 3 epochs: ~0.3-0.8 (good)
- Below 0.2: possible overfitting on small dataset